In [7]:
import pandas as pd 
import logging 
import duckdb 
 
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    filename='dataLoading.log'
)
logger = logging.getLogger(__name__)

In [8]:
### loading csv files for each city, from two time intervals + energy data 
c59 = pd.read_csv('data/cville59.csv', low_memory=False)
c918 = pd.read_csv('data/cville918.csv', low_memory=False)
d59 = pd.read_csv('data/dulles59.csv', low_memory=False)
d918 = pd.read_csv('data/dulles918.csv', low_memory=False)
nor59 = pd.read_csv('data/nor59.csv', low_memory=False)
nor918 = pd.read_csv('data/nor918.csv', low_memory=False)
ly59 = pd.read_csv('data/ly59.csv', low_memory=False)
ly918 = pd.read_csv('data/ly918.csv', low_memory=False)

dom = pd.read_csv('data/DOM_hourly.csv')

logger.info('CSV files loaded successfully')

In [9]:
### combine the two time intervals for each city into one df, sort by Date, and reset index 
cville = pd.concat([c59, c918], ignore_index=True).sort_values(by='DATE').reset_index(drop=True)
dulles = pd.concat([d59, d918], ignore_index=True).sort_values(by='DATE').reset_index(drop=True)
norfolk = pd.concat([nor59, nor918], ignore_index=True).sort_values(by='DATE').reset_index(drop=True)
lynchburg = pd.concat([ly59, ly918], ignore_index=True).sort_values(by='DATE').reset_index(drop=True)

logger.info ('Individual city dataframes created')

In [12]:
### Clean up columns and add Entry_ID as Primary Key 
dfs = {"cville": cville,"dulles": dulles, "norfolk": norfolk,"lynchburg": lynchburg}

# for df name and df in dfs dict 
for name, d in dfs.items():
    # drop unwanted columns 
    d = d.drop(columns=['BackupDirection', 'BackupDistance', 'BackupDistanceUnit', 'BackupElements', 'BackupElevation', 'BackupElevationUnit', 'BackupEquipment', 'BackupLatitude', 'BackupLongitude', 'BackupName',
                        'AWND', 'CDSD', 'CLDD', 'DSNW', 'DYHF', 'DYTS', 'HDSD', 'HTDD','WindEquipmentChangeDate'])

    # drop nas in this columns
    d2 = d.dropna(subset=['DailyAverageDryBulbTemperature']).copy()
    # ensure proper datetime format 
    d2['DATE'] = pd.to_datetime(d2['DATE'])
    d2['DATE'] = d2['DATE'].dt.floor('h')


    # map station numbers to names
    d2['STATION'] = d2['STATION'].replace({
        72403093738: 'Dulles',
        72410013733: 'Lynchburg',
        72308013737: 'Norfolk',
        72401693736: 'Charlottesville'
    })

    # create unique entry ID by combining station name and date
    d2['Entry_ID'] = (d2['STATION'].astype(str) + '_' + pd.to_datetime(d2['DATE']).dt.strftime('%Y-%m-%d_%H:%M')) 

    print(d2['STATION'].value_counts())

    # save to parquet   
    d2.to_parquet(f"data/{name}.parquet", engine='pyarrow', index=False)
    logger.info(f"{name} saved as parquet")
# save dom_hourly as parquet
dom.to_parquet("data/DOM_hourly.parquet", engine='pyarrow', index=False) 
logger.info("DOM_hourly saved as parquet")

STATION
Charlottesville    4731
Name: count, dtype: int64
STATION
Dulles    4749
Name: count, dtype: int64
STATION
Norfolk    4749
Name: count, dtype: int64
STATION
Lynchburg    4742
Name: count, dtype: int64


In [13]:
### save parquet files to duckdb database 
con = None 
try:

    # establish connection and create if doesn't exist 
    con = duckdb.connect(database='project1.db', read_only=False) 
    logger.info("Connected to duckdb instance.") 

    # load each csv into duckdb as a table
    con.execute(f"""
        DROP TABLE IF EXISTS cville;
                
        CREATE TABLE cville AS
        SELECT * FROM read_parquet('data/cville.parquet');
    """)
    con.execute(f"""
        DROP TABLE IF EXISTS dulles;
                
        CREATE TABLE dulles AS
        SELECT * FROM read_parquet('data/dulles.parquet');
    """)
    con.execute(f"""
        DROP TABLE IF EXISTS lynchburg;
        CREATE TABLE lynchburg AS
        SELECT * FROM read_parquet('data/lynchburg.parquet');
    """)
    con.execute(f"""
        DROP TABLE IF EXISTS norfolk;
        CREATE TABLE norfolk AS
        SELECT * FROM read_parquet('data/norfolk.parquet');
    """)
    con.execute(f"""
        DROP TABLE IF EXISTS dom;
        CREATE TABLE dom AS
        SELECT * FROM read_parquet('data/DOM_hourly.parquet');
    """)

    logger.info("All data loaded into duckdb.")

except Exception as e:
    logger.error(f"An error occurred: {e}")

finally:
    if con:
        con.close()
        logger.info("Duckdb connection closed.")